# 9.1 Probability for AI agents: practice notebook

Measure belief with numbers. Answer questions from a full joint table, turn causes into diagnoses with Bayes’ rule, and use conditional independence to build a naive Bayes classifier.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Queries from the dentist joint table

In [ ]:
# The full joint distribution of the dentist example (AIMA figure 13.3), as a dictionary.
# A key is (toothache, catch, cavity).
P = {(True, True, True): 0.108, (True, False, True): 0.012,
     (False, True, True): 0.072, (False, False, True): 0.008,
     (True, True, False): 0.016, (True, False, False): 0.064,
     (False, True, False): 0.144, (False, False, False): 0.576}

print("total:", round(sum(P.values()), 3))

# Marginal: add up every world where cavity is true.
p_cavity = 0
for (t, c, cav), p in P.items():
    if cav:
        p_cavity = p_cavity + p
print("P(cavity) =", round(p_cavity, 3))

# Conditional: keep the worlds with a toothache, then normalise.
yes = 0
no = 0
for (t, c, cav), p in P.items():
    if t and cav:
        yes = yes + p
    if t and not cav:
        no = no + p
alpha = 1 / (yes + no)
print("P(toothache) =", round(yes + no, 3), " alpha =", round(alpha, 3))
print("P(cavity | toothache) =", round(alpha * yes, 3))

### Bayes' rule for a medical test

In [ ]:
# Bayes' rule for a medical test: P(disease | positive) = P(positive | disease) P(disease) / P(positive)
prior = 0.01          # 1 person in 100 has the disease
sens = 0.90           # P(positive | disease)
false_pos = 0.09      # P(positive | no disease)

p_pos = sens * prior + false_pos * (1 - prior)    # total probability
post = sens * prior / p_pos
print("P(positive) =", round(p_pos, 4))
print("P(disease | positive) =", round(post, 4))

# A second, independent positive test: yesterday's posterior is today's prior.
p_pos2 = sens * post + false_pos * (1 - post)
post2 = sens * post / p_pos2
print("after two positive tests:", round(post2, 4))

# The same in odds form: posterior odds = prior odds x likelihood ratio.
odds = prior / (1 - prior)
lr = sens / false_pos
print("likelihood ratio:", round(lr, 2), " posterior odds:", round(odds * lr, 4), " ->", round(odds * lr / (1 + odds * lr), 4))

### A naive Bayes spam filter

In [ ]:
# A naive Bayes spam filter with add one (Laplace) smoothing, trained on six short messages.
train = [("win money now", "spam"), ("cheap money offer", "spam"), ("win a prize now", "spam"),
         ("meeting at noon", "ham"), ("project meeting notes", "ham"), ("lunch at noon", "ham")]

count = {"spam": 0, "ham": 0}
words = {"spam": {}, "ham": {}}
vocab = []
for text, label in train:
    count[label] = count[label] + 1
    for w in text.split():
        words[label][w] = words[label].get(w, 0) + 1
        if w not in vocab:
            vocab.append(w)

def score(text, label):
    total = sum(words[label].values())
    p = count[label] / len(train)                       # prior P(label)
    for w in text.split():
        p = p * (words[label].get(w, 0) + 1) / (total + len(vocab))   # P(word | label), smoothed
    return p

for msg in ["win money", "meeting at noon", "cheap lunch"]:
    s, h = score(msg, "spam"), score(msg, "ham")
    print(msg, "-> P(spam | words) =", round(s / (s + h), 3))

## Your turn

### Task 1: A marginal from the joint table

JOINT maps (toothache, catch, cavity) to a probability. Write `marginal(joint, i)` that returns the probability that variable number i (0, 1 or 2) is true.

Hint: Loop over the keys and add the probability when position i is True.

In [ ]:
JOINT = {(True, True, True): 0.108, (True, False, True): 0.012,
         (False, True, True): 0.072, (False, False, True): 0.008,
         (True, True, False): 0.016, (True, False, False): 0.064,
         (False, True, False): 0.144, (False, False, False): 0.576}   # key: (toothache, catch, cavity)

def marginal(joint, i):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('round(marginal(JOINT, 2), 3)', 0.2), ('round(marginal(JOINT, 0), 3)', 0.2), ('round(marginal(JOINT, 1), 3)', 0.34)])

### Task 2: A conditional probability from the joint table

Write `conditional(joint, q, e, v)` that returns P(variable q is true | variable e has value v).

Hint: Add up the worlds that match the evidence, and separately the ones that also have q true. Divide.

In [ ]:
JOINT = {(True, True, True): 0.108, (True, False, True): 0.012,
         (False, True, True): 0.072, (False, False, True): 0.008,
         (True, True, False): 0.016, (True, False, False): 0.064,
         (False, True, False): 0.144, (False, False, False): 0.576}   # key: (toothache, catch, cavity)

def conditional(joint, q, e, v):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('round(conditional(JOINT, 2, 0, True), 3)', 0.6), ('round(conditional(JOINT, 2, 1, True), 3)', 0.529), ('round(conditional(JOINT, 1, 2, True), 3)', 0.9), ('round(conditional(JOINT, 2, 0, False), 3)', 0.1)])

### Task 3: Bayes' rule for a test

Write `posterior(prior, sens, fpr)` that returns P(ill | positive) given P(ill) = prior, P(positive | ill) = sens and P(positive | well) = fpr.

Hint: P(positive) = sens × prior + fpr × (1 − prior).

In [ ]:
def posterior(prior, sens, fpr):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('round(posterior(0.01, 0.9, 0.09), 4)', 0.0917), ('round(posterior(0.5, 0.9, 0.1), 4)', 0.9), ('round(posterior(0.001, 0.99, 0.01), 4)', 0.0902), ('round(posterior(0.2, 1.0, 0.0), 4)', 1.0)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: A marginal from the joint table

P(catch) = 0.108 + 0.072 + 0.016 + 0.144 = 0.34.

In [ ]:
JOINT = {(True, True, True): 0.108, (True, False, True): 0.012,
         (False, True, True): 0.072, (False, False, True): 0.008,
         (True, True, False): 0.016, (True, False, False): 0.064,
         (False, True, False): 0.144, (False, False, False): 0.576}   # key: (toothache, catch, cavity)

def marginal(joint, i):
    total = 0
    for world in joint:
        if world[i]:
            total = total + joint[world]
    return total

In [ ]:
check([('round(marginal(JOINT, 2), 3)', 0.2), ('round(marginal(JOINT, 0), 3)', 0.2), ('round(marginal(JOINT, 1), 3)', 0.34)])

### Solution to task 2: A conditional probability from the joint table

Test 4: P(cavity | no toothache) = (0.072 + 0.008) ÷ 0.8 = 0.1.

In [ ]:
JOINT = {(True, True, True): 0.108, (True, False, True): 0.012,
         (False, True, True): 0.072, (False, False, True): 0.008,
         (True, True, False): 0.016, (True, False, False): 0.064,
         (False, True, False): 0.144, (False, False, False): 0.576}   # key: (toothache, catch, cavity)

def conditional(joint, q, e, v):
    both = 0
    given = 0
    for world in joint:
        if world[e] == v:
            given = given + joint[world]
            if world[q]:
                both = both + joint[world]
    return both / given

In [ ]:
check([('round(conditional(JOINT, 2, 0, True), 3)', 0.6), ('round(conditional(JOINT, 2, 1, True), 3)', 0.529), ('round(conditional(JOINT, 1, 2, True), 3)', 0.9), ('round(conditional(JOINT, 2, 0, False), 3)', 0.1)])

### Solution to task 3: Bayes' rule for a test

Test 3: a 99 percent accurate test for a disease of 1 in 1000 still gives only about 9 percent.

In [ ]:
def posterior(prior, sens, fpr):
    p_pos = sens * prior + fpr * (1 - prior)
    return sens * prior / p_pos

In [ ]:
check([('round(posterior(0.01, 0.9, 0.09), 4)', 0.0917), ('round(posterior(0.5, 0.9, 0.1), 4)', 0.9), ('round(posterior(0.001, 0.99, 0.01), 4)', 0.0902), ('round(posterior(0.2, 1.0, 0.0), 4)', 1.0)])